<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex02-pytorch-and-autograd/Ex02_01_tensors.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference text — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_2 · Notebook 01 — tensors

What a tensor is, how it differs from a NumPy array, and what `requires_grad`
does. No derivatives are computed here; that is notebook 02. This notebook is
the vocabulary you need to read notebook 02 without stopping.

## The one-sentence version

A tensor is a NumPy array that can live on a GPU and that can remember what
happened to it.

The first half of that sentence will not matter to you in this course — every
exercise runs on a CPU. The second half is the whole of Part 2.

## What transfers directly from Ex_1

All of it. Shape, dtype, indexing, slicing, boolean masks, broadcasting — the
rules are identical, and PyTorch adopted them deliberately so that NumPy code
would read the same way. If you are comfortable with Ex_1 notebook 02, you
already know most of this notebook and the sections below are mainly a
translation table.

What is new is at the end: `requires_grad`, `grad_fn`, and the three ways to
accidentally cut a tensor out of the graph. Read that section twice, because
every autograd problem you will have starts there.

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_2_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex02-pytorch-and-autograd/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup -------------------------------------------------------------
# Needs Ex_2_core.py alongside this notebook.
import os
for f in ("Ex_2_core.py",):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from Ex_2_core import *                             # noqa: F401,F403
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt

set_seed(88)
torch.set_printoptions(precision=4, sci_mode=False)
print("setup complete | device:", DEVICE)

## 1 · Creating tensors, and the dtype question

The constructors mirror NumPy's, with the same names in most cases:

| NumPy | PyTorch |
|---|---|
| `np.array(x)` | `torch.tensor(x)` |
| `np.zeros((2, 3))` | `torch.zeros(2, 3)` — note: not a tuple |
| `np.ones`, `np.full` | `torch.ones`, `torch.full` |
| `np.linspace(a, b, n)` | `torch.linspace(a, b, n)` |
| `np.arange(a, b, s)` | `torch.arange(a, b, s)` |
| `np.random.normal(size=s)` | `torch.randn(*s)` |
| `np.random.uniform(size=s)` | `torch.rand(*s)` |

The shape is passed as separate arguments rather than as a tuple —
`torch.zeros(2, 3)`, not `torch.zeros((2, 3))` — although the tuple form is
also accepted.

**The dtype question is not cosmetic.** PyTorch's default floating type is
`float32`; NumPy's is `float64`. Build a tensor from a NumPy array and you get
`float64`; build one from Python floats and you get `float32`; then combine
them and, depending on the operation, you get either a silent promotion or a
`RuntimeError` about expected scalar types. The defence is to say what you
mean:

```python
torch.tensor(x, dtype=torch.float32)
torch.as_tensor(x, dtype=DTYPE, device=DEVICE)
```

`DTYPE` and `DEVICE` come from the core module and are `torch.float32` and
`cpu`. Every exercise in this course, and in Part 2, writes them out.

Why `float32` at all, when `float64` is more accurate? Because it is twice as
fast and half the memory, and neural-network training does not need the
precision — the gradient noise from a finite dataset swamps the difference. The
exception is a convergence study, where you are measuring an error of 1e-10 and
the arithmetic itself becomes the limiting factor. Notebook 02 switches to
`float64` in exactly one section, for exactly that reason.

In [ ]:
from_python = torch.tensor([1.0, 2.0, 3.0])
from_numpy = torch.tensor(np.array([1.0, 2.0, 3.0]))
stated = torch.tensor(np.array([1.0, 2.0, 3.0]), dtype=DTYPE)

print("  from Python floats :", from_python.dtype)
print("  from a NumPy array :", from_numpy.dtype, "  <- float64, silently")
print("  dtype stated       :", stated.dtype)

print()
try:
    layer = nn.Linear(3, 1)
    layer(from_numpy.reshape(1, 3))
except RuntimeError as exc:
    print("  feeding a float64 tensor to a float32 layer:")
    print("   ", str(exc)[:110], "...")

**What you should see.** `torch.float32`, then `torch.float64`, then
`torch.float32` again; and then a `RuntimeError` complaining about mismatched
scalar types. The exact wording moves between torch versions — *expected scalar
type Float but found Double* is the usual one — and the vocabulary is inherited
from C: **Double** means `float64` and **Float** means `float32`.

That error, in that exact form, is the one you will meet when you load
measurement data from a file with NumPy and feed it to a network. The fix is
always the same: state the dtype when you create the tensor.

### Your turn: build three tensors

Build these, with exactly these names and properties:

| name | requirement |
|---|---|
| `t_grid` | 21 points from 0.0 to 1.0 inclusive, as a **column**: shape `(21, 1)`, dtype `float32` |
| `ones_col` | a column of six ones, shape `(6, 1)`, dtype `float32` |
| `t_from_np` | the NumPy array `np_data` below, as a `float32` tensor of shape `(5, 1)` |

Two points of technique.

**The column shape.** A network built from `nn.Linear` expects its input to
have a trailing feature axis: `(n_samples, n_features)`. With one input
variable that means `(n, 1)`, not `(n,)`. Every coordinate tensor in this
course and in the whole of Part 2 has that shape, and the reason is exactly the
`(N,)` against `(N, 1)` trap you met in Ex_1 — by never letting a `(n,)`
coordinate exist, the trap has no opportunity to fire. Get the column with
`.reshape(-1, 1)`.

**The conversion.** `torch.tensor(np_data, dtype=torch.float32)` copies;
`torch.from_numpy(np_data)` shares memory with the array and keeps its dtype.
Use the first here. Sharing memory is occasionally useful and is a good way to
be surprised later by a tensor that changed when you edited an array.

In [ ]:
np_data = np.linspace(0.0, 2.0, 5)         # float64, shape (5,)
print("np_data:", np_data.dtype, np_data.shape)

# TODO: build t_grid, ones_col and t_from_np as described above
raise NotImplementedError("build the three tensors: t_grid, ones_col, t_from_np")

In [ ]:
check_shape("t_grid", t_grid, (21, 1))
check("t_grid ends", [t_grid[0, 0].item(), t_grid[-1, 0].item()], [0.0, 1.0])
print("  t_grid dtype:", t_grid.dtype, "(expected torch.float32)")

check_shape("ones_col", ones_col, (6, 1))
check("ones_col values", ones_col, torch.ones(6, 1))

check_shape("t_from_np", t_from_np, (5, 1))
print("  t_from_np dtype:", t_from_np.dtype, "(expected torch.float32)")
check("t_from_np values", t_from_np, torch.tensor(np_data, dtype=DTYPE).reshape(-1, 1))

**What you should see.** Every check `PASS`, and both dtypes reported as
`torch.float32`. A `FAIL` on a shape of `(21,)` means you built a row and
forgot the `.reshape(-1, 1)`.

## 2 · Device

A tensor lives somewhere — main memory, or a particular GPU. Operations require
their operands to be in the same place, and moving one is explicit:

```python
t = t.to(DEVICE)
model = model.to(DEVICE)
```

In this course `DEVICE` is the CPU, because every problem is small. In Part 2
the same variable is written the standard way:

```python
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
```

which runs on a GPU when there is one and on the CPU otherwise, without any
other line of code changing. That is the whole reason the constant exists
rather than the string `"cpu"` being written in forty places.

The error you get for mismatched devices is *Expected all tensors to be on the
same device*, and it is not subtle. The one to watch for is a model moved to
the GPU while its input data was not.

In [ ]:
t = torch.ones(3)
print("  t.device            ", t.device)
print("  after .to(DEVICE)   ", t.to(DEVICE).device)
print()
print("  a GPU is", "available" if torch.cuda.is_available() else "not available",
      "- and nothing in this course needs one")

**What you should see.** `cpu` twice, and a note that no GPU is available.
Nothing here depends on it.

## 3 · Shapes

The operations you need, and their NumPy equivalents:

| | |
|---|---|
| `t.shape` | a `torch.Size`, which is a tuple — `tuple(t.shape)` prints cleanly |
| `t.reshape(a, b)` | as NumPy; `-1` means "work it out" |
| `t.view(a, b)` | the same, but only on contiguous memory and never copies |
| `t.unsqueeze(1)` | insert an axis of length 1 — NumPy's `[:, None]` |
| `t.squeeze()` | remove every axis of length 1 |
| `t.T` / `t.transpose(0, 1)` | transpose |
| `t.sum(dim=0)` | NumPy's `axis=` is spelled `dim=` |
| `t.sum(dim=0, keepdim=True)` | NumPy's `keepdims` loses its s |

`reshape` versus `view` is a distinction you can mostly ignore: `reshape` does
what you want in every case, at the cost of copying when it has to. Use it.

**Broadcasting is identical.** Same rule, same alignment from the right, same
`(n,)` against `(n, 1)` trap producing an `(n, n)` matrix in silence. The one
new thing is that PyTorch also broadcasts in matrix multiplication over leading
batch axes, which you will not need until L5.

The `squeeze()` habit deserves a warning. `t.squeeze()` removes *every* axis of
length 1, so a `(1, 1)` tensor becomes a scalar and a batch of one sample
loses its batch axis. When you know which axis you mean, say so:
`t.squeeze(1)`.

In [ ]:
v = torch.linspace(0.0, 3.0, 4)

print("  v                 ", tuple(v.shape))
print("  v.reshape(-1, 1)  ", tuple(v.reshape(-1, 1).shape), " a column")
print("  v.unsqueeze(0)    ", tuple(v.unsqueeze(0).shape), " a row")
print("  v.reshape(2, 2)   ", tuple(v.reshape(2, 2).shape))
print("  v.reshape(2, 2).T ", tuple(v.reshape(2, 2).T.shape))

col = v.reshape(-1, 1)
row = v.reshape(1, -1)
print()
print("  col - row shape   ", tuple((col - row).shape), " <- (4, 1) against (1, 4)")
print("  v - col shape     ", tuple((v - col).shape), " <- the Ex_1 trap, in torch")

**What you should see.** The four reshapes, then two `(4, 4)` results. The
second one is the trap: subtracting a `(4, 1)` column from a `(4,)` vector
gives a matrix, in PyTorch exactly as in NumPy, with no complaint.

### Your turn: shapes and broadcasting in torch

Given `v` of shape `(6,)`, build:

| name | requirement |
|---|---|
| `v_col` | shape `(6, 1)` |
| `v_row` | shape `(1, 6)` |
| `diff` | shape `(6, 6)`, with `diff[i, j] = v[i] - v[j]` |
| `scaled` | `v_col` divided by its own maximum, shape `(6, 1)` |

`diff` is the same construction as the distance matrix in Ex_1 notebook 02, and
it is worth writing once in PyTorch so that the syntax is not new when you meet
it in a residual.

For `scaled`, `v_col.max()` returns a scalar tensor, which broadcasts against
anything. Watch the order of indices in `diff`: the row index varies along the
first axis, so the column tensor supplies `i` and the row tensor supplies `j`.

In [ ]:
v = torch.tensor([2.0, 5.0, 1.0, 8.0, 3.0, 4.0])

# TODO: build v_col, v_row, diff and scaled
raise NotImplementedError("v_col (6,1), v_row (1,6), diff (6,6) with diff[i,j] = v[i]-v[j], scaled")

In [ ]:
check_shape("v_col", v_col, (6, 1))
check_shape("v_row", v_row, (1, 6))
check_shape("diff", diff, (6, 6))
check("diff[0, 3]", diff[0, 3], torch.tensor(2.0 - 8.0))
check("diff is antisymmetric", diff, -diff.T)
check("zero diagonal", torch.diagonal(diff), torch.zeros(6))
check_shape("scaled", scaled, (6, 1))
check("scaled max is 1", scaled.max(), torch.tensor(1.0))

**What you should see.** Every check `PASS`. If `diff[0, 3]` comes out as
`+6.0` rather than `-6.0` you have the column and row the wrong way round —
which is not fatal here, but in a residual it is the difference between a
gradient and its negative.

## 4 · Matrix multiplication, and what a layer is

`*` is elementwise. Matrix multiplication is `@`, or `torch.matmul`. This
catches people arriving from MATLAB, where `*` is the matrix product and `.*`
is elementwise — the convention is exactly reversed.

A fully connected layer is one matrix multiplication and one addition:

    y = x W^T + b

with `x` of shape `(n_samples, n_in)`, `W` of shape `(n_out, n_in)`, `b` of
shape `(n_out,)`, and `y` of shape `(n_samples, n_out)`. The transpose is a
storage convention, not mathematics: `nn.Linear` stores its weight as
`(n_out, n_in)` because that is the layout its backward pass prefers.

That is genuinely all a linear layer is. The next task is to write one by hand
and check it against `nn.Linear`, because a network you have taken apart once
is a network you will debug rather than pray to.

In [ ]:
layer = nn.Linear(3, 2)
x = torch.tensor([[1.0, 2.0, 3.0],
                  [0.5, 0.0, -1.0]])

print("  x       ", tuple(x.shape))
print("  weight  ", tuple(layer.weight.shape), " (n_out, n_in)")
print("  bias    ", tuple(layer.bias.shape))
print("  layer(x)", tuple(layer(x).shape))
print()
print("  weight requires_grad:", layer.weight.requires_grad,
      " <- parameters are tracked automatically")

In [ ]:
# TODO: compute `y_manual` from layer.weight, layer.bias and x, using @ and +
raise NotImplementedError("y_manual = x @ W^T + b, with the shapes above")

In [ ]:
check_shape("y_manual", y_manual, (2, 2))
check("y_manual matches nn.Linear", y_manual, layer(x), tol=1e-6)
print()
print("  the layer has", count_parameters(layer), "parameters: 3x2 weights plus 2 biases")

**What you should see.** Two `PASS` lines and a parameter count of 8.

A note on the tolerance. `check` here uses 1e-6 rather than the exact equality
you might expect, because the two computations sum the same products in
different orders and floating-point addition is not associative. Bitwise
equality is the wrong thing to test for in floating point, and asking for it is
a common way to write a test that fails for no reason.

## 5 · Reductions

`sum`, `mean`, `max`, `min`, `std` — all take `dim=` where NumPy takes `axis=`,
and `keepdim=` where NumPy takes `keepdims=`. The behaviour is the same: the
named axis is the one that disappears, and `keepdim=True` leaves it in place
with length 1 so the result broadcasts back against the original.

One difference worth knowing: `t.max()` with no arguments returns a scalar,
but `t.max(dim=0)` returns a **named tuple** of `(values, indices)`. So

```python
vals, idx = t.max(dim=0)
```

and if you write `t.max(dim=0)` where you expected a tensor you will get a
confusing error two lines later. `torch.amax(t, dim=0)` returns just the
values, if you want them alone.

In [ ]:
a = torch.arange(12.0).reshape(3, 4)

print("  a.sum()               ", a.sum().item())
print("  a.sum(dim=0)          ", a.sum(dim=0), tuple(a.sum(dim=0).shape))
print("  a.sum(dim=1)          ", a.sum(dim=1), tuple(a.sum(dim=1).shape))
print("  a.sum(dim=1, keepdim) ", tuple(a.sum(dim=1, keepdim=True).shape))
print()
vals, idx = a.max(dim=1)
print("  a.max(dim=1) values   ", vals)
print("  a.max(dim=1) indices  ", idx)

### Your turn: a mean squared error, by hand

Every loss in this course is a mean of squares. Write one.

Given `pred` and `target`, both of shape `(n, 1)`, compute:

| name | requirement |
|---|---|
| `residual` | `pred - target`, shape `(n, 1)` |
| `mse` | the mean of the squared residual, a **scalar tensor** |
| `rmse` | its square root, a scalar tensor |

A scalar tensor is one with shape `()` — no axes at all. That is what
`.mean()` with no `dim` returns, and it is what a loss has to be: `backward()`
requires a scalar, because a derivative with respect to many outputs at once is
a Jacobian rather than a gradient.

Do not use `torch.nn.functional.mse_loss` here. Writing the two lines by hand
once is the point, and in Part 2 you will be writing losses that no library
function computes.

In [ ]:
set_seed(88)
target = torch.randn(20, 1)
pred = target + 0.1 * torch.randn(20, 1)

# TODO: compute residual, mse and rmse
raise NotImplementedError("residual = pred - target, mse = mean of its square, rmse = sqrt(mse)")

In [ ]:
check_shape("residual", residual, (20, 1))
print("  mse shape :", tuple(mse.shape), "(expected () - a scalar tensor)")
check("mse", mse, ((pred - target) ** 2).mean())
check("rmse", rmse, ((pred - target) ** 2).mean().sqrt())
print(f"  rmse = {rmse.item():.4f}, and the noise added was 0.1000")

**What you should see.** Two `PASS` lines, a shape of `()` for the loss, and an
RMSE near 0.1 — recovering the noise, as it did in Ex_1. If `mse` has shape
`(1, 1)` you reduced along one axis only; use `.mean()` with no arguments.

## 6 · `requires_grad`, and the graph

This is the section that matters, and it is short because the mechanism is
simple.

A tensor has a flag, `requires_grad`. When it is `True`, PyTorch records every
operation the tensor takes part in, building a **graph** whose nodes are
operations and whose edges are tensors. Each result of such an operation
carries a `grad_fn`, which is the node that produced it and knows how to
differentiate it.

Three terms, and then you have the whole vocabulary:

- **leaf**: a tensor you created directly, rather than one computed from
  others. Your inputs and your network's parameters are leaves. `t.is_leaf`
  tells you.
- **`grad_fn`**: `None` for a leaf; otherwise the operation that produced the
  tensor. `AddBackward0`, `MulBackward0`, `TanhBackward0` — the name is the
  forward operation with `Backward` appended.
- **`.grad`**: where `backward()` deposits the accumulated gradient, on leaves
  only. It is `None` until something has been backpropagated into it.

Parameters of an `nn.Module` have `requires_grad=True` automatically — that is
what makes them trainable. An input does not, unless you say so, and saying so
is exactly what notebook 02 is about.

In [ ]:
x = torch.tensor([2.0], requires_grad=True)
y = x ** 2 + 3 * x

describe_tensor(x, "x  (a leaf, marked)")
print()
describe_tensor(y, "y = x^2 + 3x  (computed)")

print()
print("  y.grad_fn.next_functions:", y.grad_fn.next_functions)
print("  ... the graph, one level down")

**What you should see.** `x` with `requires_grad True`, `is_leaf True`, and
`grad_fn None`; `y` with `requires_grad True`, `is_leaf False`, and a `grad_fn`
of `<AddBackward0 ...>`. The last line shows the two branches feeding that
addition.

A leaf that requires grad is an input. A non-leaf that requires grad is
something computed from one. That distinction is all you need.

### The three ways to lose the graph

Every "why is my gradient `None`?" question has one of these three answers.

**`.detach()`** returns a tensor with the same values and no history. It is the
correct tool when you want a number for plotting or logging, and a bug when you
leave it in the middle of a computation you intend to differentiate.

**`.item()` and `.numpy()`** leave the tensor world entirely, producing a
Python float or a NumPy array. Anything computed from them is invisible to
autograd. Converting to NumPy midway through a loss function is the classic
version of this mistake.

**`with torch.no_grad():`** switches recording off for a block. It is what you
use around evaluation code, and it makes everything computed inside it
untrackable. If you write your validation pass inside `no_grad` and then try to
differentiate its output, nothing happens.

The last task is to demonstrate the first one deliberately.

In [ ]:
x = torch.tensor([3.0], requires_grad=True)

# TODO: build two tensors from x:
#         `tracked`   = x ** 2, which still carries a grad_fn
#         `untracked` = the same value, detached, with requires_grad False
raise NotImplementedError("tracked = x ** 2; untracked = the same value with the graph cut")

In [ ]:
print("  tracked   requires_grad:", tracked.requires_grad, " grad_fn:", tracked.grad_fn)
print("  untracked requires_grad:", untracked.requires_grad, " grad_fn:", untracked.grad_fn)
check("same value", tracked, untracked)

ok = (tracked.requires_grad and tracked.grad_fn is not None
      and not untracked.requires_grad and untracked.grad_fn is None)
print()
print("  PASS" if ok else "  FAIL", "- one tracked, one not, same numbers")

print()
with torch.no_grad():
    inside = x ** 2
print("  computed inside no_grad(): requires_grad =", inside.requires_grad,
      " <- the third way")

**What you should see.** `tracked` with `requires_grad True` and a
`PowBackward0`; `untracked` with `False` and `None`; the values equal; `PASS`;
and the `no_grad` result also untracked.

Both tensors hold 9.0. Only one of them can tell you that its derivative with
respect to `x` is 6.

## 7 · Translation table

Keep this. It answers most "how do I do X in torch" questions.

| NumPy | PyTorch |
|---|---|
| `np.array(x)` | `torch.tensor(x)`, or `torch.as_tensor` to avoid a copy |
| `a.astype(np.float32)` | `a.to(torch.float32)` |
| `a.shape` | `a.shape`, a `torch.Size` |
| `a.reshape(...)` | `a.reshape(...)` or `a.view(...)` |
| `a[:, None]` | `a.unsqueeze(1)` — both notations work in torch |
| `a.sum(axis=0)` | `a.sum(dim=0)` |
| `keepdims=True` | `keepdim=True` |
| `a @ b` | `a @ b` — same |
| `np.concatenate` | `torch.cat` |
| `np.stack` | `torch.stack` |
| `np.where` | `torch.where` |
| `np.exp`, `np.sin` | `torch.exp`, `torch.sin` |
| — | `a.detach()`, `a.item()`, `a.requires_grad` |

**One rule for the whole table.** Inside anything you intend to differentiate,
use the `torch` version of every mathematical function. `np.sin(tensor)` will
often appear to work and will silently produce something autograd cannot
follow, or will raise. `torch.sin` is differentiable; `np.sin` is not.

## What you have done

You can create tensors with a stated dtype and device; you know why every
coordinate in this course has shape `(n, 1)`; you can reshape, broadcast and
reduce; you have written a linear layer by hand and checked it against
`nn.Linear`; you have written a mean squared error by hand; and you can say
what `requires_grad`, `is_leaf` and `grad_fn` mean and name three ways to lose
the graph.

## Next

`Ex02_02_autograd_by_hand.ipynb` — the most important notebook in Part 1.